# Proceso de Machine Learning

## Preparación de datos

In [20]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score



# Leer el archivo CSV
df = pd.read_csv('https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv', sep=',') # Este archivo CSV contiene comas como separadores
print(df.head())
print(df.columns) 

   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              18  United-States

## Exploración y limpieza de datos

Comprobamos las dimensiones del dataframe

In [21]:
print(f"Dimensiones del dataframe: {df.shape}")
print(df.info())

Dimensiones del dataframe: (32561, 15)
<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB
None


Procedemos a contabilizar los nulos y únicos: 

In [22]:
print(f"Valores null por columna: \n{df.isnull().sum()}")
print(f"Valores unicos por columna: \n{df.nunique()}")

Valores null por columna: 
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64
Valores unicos por columna: 
age                  73
workclass             9
fnlwgt            21648
education            16
education.num        16
marital.status        7
occupation           15
relationship          6
race                  5
sex                   2
capital.gain        119
capital.loss         92
hours.per.week       94
native.country       42
income                2
dtype: int64


Borramos los duplicados y eliminamos columnas que no aporten información relevante

In [23]:
df = df.drop_duplicates()
print(f"Dimensiones del dataframe: {df.shape}")

df.drop(['workclass', 'capital.gain', 'capital.loss', 'fnlwgt', 'education.num', 'relationship', 'race'], axis=1, inplace=True)
print(f"Dimensiones del dataframe: {df.shape}")

Dimensiones del dataframe: (32537, 15)
Dimensiones del dataframe: (32537, 8)


Transformación de variables mal codificadas

In [24]:
df = df.replace("?", np.nan, inplace=True)

# Reemplazamos los nan por valores de moda
for col in df.columns:
      df[col] = df[col].fillna(df[col].mode()[0])

In [25]:
df['income'] = df['income'].str.strip()
df['high_income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

Para este caso, se busca recomendar si el usuario va a ganar mas de 50k$ al año o si puede mejorar alguna característica para estar más cerca de lograrlo.  
Procedemos a separar variables numéricas y categóricas

In [26]:
var_numericas = ['age', 'hours.per.week']
var_categoricas = ['education', 'marital.status', 'occupation', 'sex', 'native.country']

X = df.drop(["income", "high_income"], axis=1)
y = df["high_income"]

transformador_numerico = StandardScaler()
transformador_categorico = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, var_numericas),
        ("cat", transformador_categorico, var_categoricas),
    ]
)

Procedemos a crear los conjuntos de datos para entrenar el modelo

In [27]:
pipe = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

X_train_trans = pipe.fit_transform(X_train)
X_test_trans = pipe.transform(X_test)

sim_matrix = cosine_similarity(X_test_trans, X_train_trans)


k = 5
y_pred = []

for i in range(sim_matrix.shape[0]):
    sim_scores = sim_matrix[i]
    
    # índices de los k más similares
    top_k_idx = np.argsort(sim_scores)[-k:]
    
    # etiquetas de esos vecinos
    top_k_labels = y_train.iloc[top_k_idx]
    
    # votación mayoritaria
    pred = int(top_k_labels.mean() > 0.5)
    y_pred.append(pred)

y_pred = np.array(y_pred)


accuracy = accuracy_score(y_test, y_pred)
print("El valor de accuracy es: ", accuracy)

El valor de accuracy es:  0.8183773816840811


In [28]:
def recomendar_usuario(k, perfil_usuario):

    if k <= 0 or k is None or k > len(X_train):
        k = 5
    
    if isinstance(perfil_usuario, dict):
        usuario_df = pd.DataFrame([perfil_usuario])
    elif isinstance(perfil_usuario, pd.DataFrame):
        usuario_df = perfil_usuario.copy()

    usuario_df = usuario_df.reindex(columns=X_train.columns)

    usuario_trans = pipe.transform(usuario_df)

    sim = cosine_similarity(usuario_trans, X_train_trans)[0]
    top_k_idx = np.argsort(sim)[-k:]

    pred = int(y_train.iloc[top_k_idx].mean() > 0.5)

    vecinos = X_train.iloc[top_k_idx]
    labels = y_train.iloc[top_k_idx]

    pred = int(labels.mean() > 0.5) # prediccion si gana mas o menos de 50k representado con 0 o 1 con la media de los usuarios cercanos

    vecinos_ricos = vecinos[labels == 1]

    if len(vecinos_ricos) > 0:

        recomendaciones = "Considera mejorar tu educación o cambiar de ocupación"

    else:
        recomendaciones = "No hay suficientes usuarios ricos para recomendar"

    return {
        "prediccion": ">50K" if pred == 1 else "<=50K",
        "recomendaciones": recomendaciones
    }

In [29]:
# Ejemplo: Usuario de 25 años, secundario completo, trabaja medio tiempo
perfil_usuario = {
    "age": 25,
    "education": "HS-grad",
    "marital.status": "Never-married",
    "occupation": "Sales",
    "hours.per.week": 30,
    "sex": "Male",
    "native.country": "United-States"
}

print(recomendar_usuario(50, perfil_usuario))

print(recomendar_usuario(20, df.iloc[[10]]))
print(recomendar_usuario(20, df.iloc[[20]]))
print(recomendar_usuario(20, df.iloc[[30]]))

{'prediccion': '<=50K', 'recomendaciones': 'No hay suficientes usuarios ricos para recomendar'}
{'prediccion': '<=50K', 'recomendaciones': 'Considera mejorar tu educación o cambiar de ocupación'}
{'prediccion': '<=50K', 'recomendaciones': 'Considera mejorar tu educación o cambiar de ocupación'}
{'prediccion': '<=50K', 'recomendaciones': 'Considera mejorar tu educación o cambiar de ocupación'}
